# SABR Paper Reproduction Walkthrough - Fast Presentation Demo

This notebook mirrors `paper_reproduction_walkthrough.ipynb` section by section, but uses smaller Monte Carlo budgets so it can be rerun live during a presentation.

Paper:
- Choi, Hu, Kwok, *Efficient and accurate simulation of the stochastic-alpha-beta-rho model*

Project scope preserved from the full walkthrough:
- core SABR Monte Carlo simulator
- sanity checks for limiting cases and martingale behavior
- paper-style tables and figure datasets
- quick and paper-scale validation hooks

Important: the live-demo cells below are intentionally noisier than the full paper-scale reproduction because `n_paths` and `n_repeats` are reduced. The algorithms, parameter cases, and helper functions are unchanged.


## Guided Paper Introduction

This opening section is a compact English version of the project intro. It follows the paper's logic step by step before we run the code.

### Stop 1: What is the SABR model?

The SABR model describes the joint movement of a forward price and its stochastic volatility:

$$
dF_t = \sigma_t F_t^\beta dW_t,
\qquad
\frac{d\sigma_t}{\sigma_t} = \nu dZ_t,
\qquad
dW_t dZ_t = \rho dt.
$$

In plain language:

| Symbol | Meaning |
|---|---|
| $F_t$ | Forward price, the quantity we simulate and price options on. |
| $\sigma_t$ | Stochastic volatility, itself random over time. |
| $\beta$ | Elasticity parameter: closer to 1 means lognormal-like, closer to 0 means normal-like. |
| $\nu$ | Vol-of-vol: how violently volatility itself moves. |
| $\rho$ | Correlation between price and volatility shocks. |

A key convenience is that volatility is easy to simulate exactly:

$$
\sigma_{t+h}=\sigma_t\exp\left(\nu\sqrt{h}X-\frac12\nu^2h\right),
\qquad X\sim N(0,1).
$$

So the hard part is not the volatility step. The hard part is how to simulate the next forward price accurately and fast.

### Stop 2: The two-step simulation framework

Once $\sigma_{t+h}$ is known, SABR simulation methods usually split the remaining problem into two conditional tasks:

<div style="display:flex; align-items:center; gap:10px; flex-wrap:wrap; margin:12px 0;">
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:210px;"><b>Known at time t</b><br><code>F_t, sigma_t</code></div>
  <div style="font-size:24px;">-&gt;</div>
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:220px;"><b>Exact volatility step</b><br>sample <code>sigma_{t+h}</code></div>
  <div style="font-size:24px;">-&gt;</div>
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:230px;"><b>Step 1</b><br>sample average variance <code>I_t^h</code></div>
  <div style="font-size:24px;">-&gt;</div>
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:230px;"><b>Step 2</b><br>sample next forward <code>F_{t+h}</code></div>
</div>

The normalized average variance is

$$
I_t^h = \frac{1}{\sigma_t^2 h}\int_t^{t+h}\sigma_s^2ds\;\Bigg|\;\sigma_{t+h}.
$$

Intuitively, $I_t^h$ measures how much volatility was accumulated during the step. It is scaled so that when vol-of-vol is zero, $I_t^h=1$.

### Stop 3: Special cases that explain the target

Before the general case, the paper reviews special cases where the conditional forward distribution is easier:

| Case | Why it helps |
|---|---|
| $\beta=0$ normal SABR | Conditional forward is normal, but the price can become negative. |
| $\beta=1$ lognormal SABR | Conditional forward is lognormal with an explicit conditional mean. |
| $\nu=0$ zero vol-of-vol | SABR reduces to the CEV model. |
| $\rho=0$ zero correlation | Conditional forward also has a CEV form. |

These cases reveal the central design goal: a good general SABR simulator should be fast, accurate, nonnegative for $0<\beta<1$, and should preserve the martingale condition

$$
\mathbb{E}[F_T]\approx F_0.
$$


## The Paper's Main Algorithmic Ideas

### Stop 4: Step 1 samples average variance with a shifted lognormal distribution

The paper approximates $I_t^h$ with a shifted-lognormal random variable:

$$
I_t^h \approx \mu\left[\frac16 + \frac56\exp\left(aX-\frac12a^2\right)\right],
\qquad X\sim N(0,1).
$$

The mean and variance inputs come from exact conditional moment formulas. The shift weight $5/6$ comes from the small-time limit.

Why this matters:

| Method idea | Tradeoff |
|---|---|
| Plain lognormal approximation | Fast, but only matches fewer moments. |
| Inverse transform / Fourier methods | More exact, but slow and implementation-heavy. |
| Paper's shifted lognormal | Still fast, but matches the distribution shape much better. |

This is Algorithm 1. Figure 1 later checks that the shifted-lognormal approximation tracks skewness and excess kurtosis well.

### Stop 5: Step 2 uses a martingale-preserving CEV approximation

The paper's main innovation is the conditional forward approximation. In the general correlated case, the exact conditional law of $F_{t+h}$ is not available. The paper approximates it as a CEV distribution:

$$
F_{t+h}\mid \sigma_{t+h}, I_t^h
\approx
\mathrm{CEV}_{\beta}\left(\bar F_t^h, (\rho^*)^2\sigma_t^2hI_t^h\right).
$$

The crucial object is $\bar F_t^h$, the conditional mean parameter. It is constructed so that the CEV approximation preserves the forward martingale property as much as possible.

This is the key difference from Islah's approximation. Islah's method is widely used, but it can break the martingale property, so forward-price drift accumulates over long maturities. Figure 3 later demonstrates this directly.

### Stop 6: CEV can be sampled exactly and quickly

A CEV transition is connected to a noncentral chi-square distribution. The paper uses a Gamma-Poisson-Gamma construction to sample it without root-finding.

<div style="display:flex; align-items:center; gap:10px; flex-wrap:wrap; margin:12px 0;">
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:185px;"><b>Gamma draw</b><br>test absorption at zero</div>
  <div style="font-size:24px;">-&gt;</div>
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:185px;"><b>Poisson draw</b><br>mixture index</div>
  <div style="font-size:24px;">-&gt;</div>
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:185px;"><b>Gamma draw</b><br>terminal CEV state</div>
  <div style="font-size:24px;">-&gt;</div>
  <div style="border:1px solid #888; border-radius:6px; padding:10px; background:#f7f7f7; width:185px;"><b>Transform back</b><br>obtain <code>F_{t+h}</code></div>
</div>

This is Algorithm 3. It is fast because it uses standard random variables and avoids iterative inversion.

### Stop 7: Algorithm 4 puts everything together

For each time step:

1. Sample $\sigma_{t+h}$ exactly.
2. Sample $I_t^h$ with the shifted-lognormal approximation.
3. Compute the martingale-preserving CEV mean $\bar F_t^h$.
4. Sample $F_{t+h}$ from the CEV distribution exactly.
5. Repeat until maturity.

The overall method is efficient because every step uses direct random sampling, not PDE solves, matrix operations, or inverse CDF root-finding.


## How the Numerical Sections Fit the Story

### Stop 8: What the paper's tables and figures are trying to prove

| Result | Purpose in the paper | Where we run it |
|---|---|---|
| Table 1 | Isolate the error from the shifted-lognormal average-variance approximation in the $\beta=1$ case. | Section 6 |
| Table 2 | Separate error sources across $\rho=1$, $0.75$, and $0$, and across different $\beta$ and $\nu$. | Section 7 |
| Figure 1 | Show that shifted lognormal captures higher moments of $I_t^h$ better than plain lognormal. | Section 8 |
| Tables 4-5 | Test long-maturity Case I and II option prices against FDM and analytic approximation rows. | Sections 9-10 |
| Table 6 | Compare Case III against other Monte Carlo baseline rows from the paper. | Section 11 |
| Table 7 / Figure 2 | Show the runtime versus RMS-error tradeoff. | Section 12 |
| Figure 3 | Compare the paper method against Islah and show the martingale advantage. | Section 13 |

For the live presentation, we keep the same functions, cases, and section order, but reduce path counts and repeats. That makes the notebook runnable on stage while preserving the paper's logic.


## Contents

0. [Guided Paper Introduction](#Guided-Paper-Introduction)
0. [The Paper's Main Algorithmic Ideas](#The-Paper's-Main-Algorithmic-Ideas)
0. [How the Numerical Sections Fit the Story](#How-the-Numerical-Sections-Fit-the-Story)
1. [Environment and Imports](#1.-Environment-and-Imports)
2. [Project API Overview](#2.-Project-API-Overview)
3. [Core Mathematical Formulas](#3.-Core-Mathematical-Formulas)
4. [Table 3 Cases](#4.-Table-3-Cases)
5. [Sanity Checks](#5.-Sanity-Checks)
6. [Paper Table 1](#6.-Paper-Table-1)
7. [Paper Table 2](#7.-Paper-Table-2)
8. [Paper Figure 1 Dataset](#8.-Paper-Figure-1-Dataset)
9. [Paper Table 4](#9.-Paper-Table-4)
10. [Paper Table 5](#10.-Paper-Table-5)
11. [Paper Table 6](#11.-Paper-Table-6)
12. [Paper Table 7 / Figure 2](#12.-Paper-Table-7-/-Figure-2)
13. [Paper Figure 3](#13.-Paper-Figure-3)
14. [Validation Summary](#14.-Validation-Summary)
15. [Optional CSV Export Helpers](#15.-Optional-CSV-Export-Helpers)


## 1. Environment and Imports

Run this first. It checks the working directory, imports the project module, and shows package versions when available.


In [ ]:
from pathlib import Path
import sys
import platform
import importlib
import time

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Project root:', PROJECT_ROOT)

for pkg in ['numpy', 'pandas', 'scipy', 'pyfeng', 'pytest']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}:', getattr(mod, '__version__', 'version unavailable'))


DEMO_N_PATHS_SMALL = 3_000
DEMO_N_PATHS_TABLE = 5_000
DEMO_REPEATS_SMALL = 2
DEMO_REPEATS_TABLE = 2
DEMO_TABLE7_BASE_PATHS = 5_000
DEMO_TABLE7_BENCHMARK_PATHS = 10_000
DEMO_FIGURE3_PATHS = 3_000
DEMO_FIGURE3_BENCHMARK_PATHS = 6_000

runtime_rows = []
notebook_start = time.perf_counter()

def timed(label, fn):
    start = time.perf_counter()
    value = fn()
    elapsed = time.perf_counter() - start
    runtime_rows.append({'section': label, 'seconds': elapsed})
    print(f'{label}: {elapsed:.2f} sec')
    return value

print('Fast demo mode: smaller Monte Carlo budgets for live presentation')
print('Table demos:', DEMO_N_PATHS_TABLE, 'paths x', DEMO_REPEATS_TABLE, 'repeats')


In [ ]:
from sabr_replicate import (
    FDMConfig,
    MonteCarloConfig,
    SABRParams,
    case_table_3,
    conditional_integrated_variance_moments,
    european_call_price,
    fdm_benchmark_prices,
    figure1_moment_comparison,
    figure2_runtime_tradeoff,
    finite_difference_call_price,
    martingale_test,
    raw_moments_to_central_stats,
    run_figure3_experiment,
    run_full_validation,
    run_table1_experiment,
    run_table2_experiment,
    run_table4_experiment,
    run_table5_experiment,
    run_table6_experiment,
    run_table7_experiment,
    sample_conditional_integrated_variance,
    simulate_terminal_forward,
    simulate_terminal_forward_islah,
)
import pyfeng as pf


## 2. Project API Overview

This section is only for orientation. It lists the main functions used later in the notebook.


## 3. Core Mathematical Formulas


Presentation note: the guided introduction above is designed to be enough for the live talk. This formula section is kept as a reference page so the fast demo still mirrors the original walkthrough.
This section collects the formulas that the implementation is reproducing. In the code, the same objects appear as `SABRParams`, `MonteCarloConfig`, `sample_conditional_integrated_variance`, `sample_cev_exact`, and `simulate_terminal_forward`.

### 3.1 SABR model

The SABR dynamics are

$$
dF_t = \sigma_t F_t^\beta\, dW_t,
\qquad
\frac{d\sigma_t}{\sigma_t} = \nu\, dZ_t,
\qquad
dW_t\,dZ_t = \rho\,dt .
$$

The exact volatility step over one interval of length $h$ is

$$
\sigma_{t+h}
=
\sigma_t
\exp\left(\nu\sqrt{h}X_\sigma - \frac{1}{2}\nu^2 h\right),
\qquad X_\sigma \sim N(0,1).
$$

We use

$$
\hat\nu = \nu\sqrt{h},
\qquad
\beta^* = 1-\beta,
\qquad
\rho^* = \sqrt{1-\rho^2}.
$$

### 3.2 Algorithm 1: conditional average variance

After sampling $\sigma_{t+h}$, the first hard quantity is the normalized conditional average variance

$$
I_t^h
=
\frac{1}{\sigma_t^2 h}
\int_t^{t+h} \sigma_s^2\,ds
\;\Bigg|\;\sigma_{t+h}.
$$

Let

$$
\mu = \mathbb{E}\left[I_t^h \mid \sigma_{t+h}\right],
\qquad
v =
\frac{
\sqrt{\mathrm{Var}\left(I_t^h \mid \sigma_{t+h}\right)}
}{
\mathbb{E}\left[I_t^h \mid \sigma_{t+h}\right]
}.
$$

With fixed shift

$$
\lambda = \frac{5}{6},
$$

the lognormal shape parameter is

$$
a
=
\sqrt{\log\left(1 + \frac{v^2}{\lambda^2}\right)}
=
\sqrt{\log\left(1 + \frac{36}{25}v^2\right)}.
$$

Algorithm 1 samples

$$
I_t^h
\approx
\mu
\left[
\frac{1}{6}
+
\frac{5}{6}
\exp\left(aX - \frac{1}{2}a^2\right)
\right],
\qquad X\sim N(0,1).
$$

In the implementation, `conditional_integrated_variance_moments` supplies the moments and `sample_conditional_integrated_variance` performs this shifted-lognormal sampling.

### 3.3 Algorithm 2: martingale-preserving CEV approximation

For $0<\beta<1$, conditional on $\sigma_{t+h}$ and $I_t^h$,

$$
F_{t+h}\mid \sigma_{t+h}, I_t^h
\approx
\mathrm{CEV}_\beta\left(\bar F_t^h, (\rho^*)^2\sigma_t^2 h I_t^h\right).
$$

The conditional mean is

$$
\bar F_t^h
\approx
F_t
\exp\left(
\frac{\rho(\sigma_{t+h}-\sigma_t)}{\nu F_t^{\beta^*}}
-
\frac{\rho^2\sigma_t^2 h I_t^h}{2F_t^{2\beta^*}}
\right).
$$

This construction is meant to preserve the martingale condition:

$$
\mathbb{E}[F_{t+h}\mid \sigma_{t+h}, I_t^h] = \bar F_t^h,
\qquad
\mathbb{E}[F_{t+h}] = F_t.
$$

For the lognormal special case $\beta=1$,

$$
F_{t+h}
=
\bar F_t^h
\exp\left(
\rho^*\sigma_t\sqrt{hI_t^h}X
-
\frac{1}{2}(\rho^*)^2\sigma_t^2 hI_t^h
\right),
\qquad X\sim N(0,1).
$$

### 3.4 Algorithm 3: exact CEV sampling

For
$$
F_T \sim \mathrm{CEV}_\beta(F_0, \sigma_0^2T),
$$

define
$$
\alpha = \frac{1}{2\beta^*},
\qquad
z_0 = \frac{F_0^{2\beta^*}}{(\beta^*)^2\sigma_0^2T}.
$$

Sample
$$
X \sim \Gamma(\alpha,1),
$$
where $\Gamma(\alpha,1)$ denotes a Gamma distribution with shape $\alpha$ and unit scale.

If $X \ge z_0/2$, set $F_T = 0$ (absorbing boundary). Otherwise sample
$$
N \sim \mathrm{Poisson}(z_0/2 - X),
\qquad
z_T \sim 2\Gamma(N+1,1),
$$

and return
$$
F_T = \left((\beta^*)^2\sigma_0^2Tz_T\right)^{1/(2\beta^*)}.
$$

This provides an exact sampling scheme for the CEV distribution via a Poisson–Gamma mixture representation.

Inside SABR, this sampler is applied with
$$
F_0 \leftarrow \bar F_t^h,
\qquad
\sigma_0^2T \leftarrow (\rho^*)^2\sigma_t^2hI_t^h.
$$

### 3.5 Algorithm 4: full simulation over a time grid

For each time step:

1. Sample $\sigma_{t+h}$ exactly.
2. Sample $I_t^h$ using Algorithm 1.
3. Compute $\bar F_t^h$ using Algorithm 2.
4. Sample $F_{t+h}$ using Algorithm 3.
5. Repeat until maturity $T$.

The Monte Carlo European call estimator is

$$
\widehat C(K)
=
\frac{1}{N}\sum_{i=1}^N \max(F_T^{(i)}-K,0).
$$

The reported errors are

$$
\text{bias} = \widehat C - C_{\text{benchmark}},
\qquad
\text{relative error} = \frac{\widehat C-C_{\text{benchmark}}}{C_{\text{benchmark}}},
$$

$$
\text{RMS error} = \sqrt{\text{bias}^2 + \text{stdev}^2}.
$$

The martingale check verifies numerically that

$$
\mathbb{E}[F_T] = F_0.
$$

Presentation note: these formulas correspond directly to Algorithms 1-4. The implementation uses the same objects through `sample_conditional_integrated_variance`, `sample_cev_exact`, and `simulate_terminal_forward`.


In [ ]:
api_overview = pd.DataFrame(
    [
        ('simulate_terminal_forward', 'Main SABR Monte Carlo terminal simulator using the paper scheme'),
        ('simulate_terminal_forward_islah', 'Appendix B / Islah-style comparison branch'),
        ('sample_conditional_integrated_variance', 'Algorithm 1 style conditional average-variance sampling'),
        ('conditional_integrated_variance_moments', 'Conditional raw moments of normalized average variance'),
        ('run_table1_experiment', 'Paper Table 1 dataset'),
        ('run_table2_experiment', 'Paper Table 2 dataset'),
        ('run_table4_experiment', 'Paper Table 4 dataset'),
        ('run_table5_experiment', 'Paper Table 5 dataset'),
        ('run_table6_experiment', 'Paper Table 6 dataset'),
        ('run_table7_experiment', 'Paper Table 7 / Figure 2 dataset'),
        ('run_figure3_experiment', 'Paper Figure 3 dataset'),
        ('run_full_validation', 'Repository validation harness'),
    ],
    columns=['function', 'purpose'],
)
api_overview


## 4. Table 3 Cases

These are the paper parameter presets used throughout later sections.


In [ ]:
case_df = pd.DataFrame(case_table_3()).T
case_df.index.name = 'case'
case_df


## 5. Sanity Checks

These cells verify model-level behavior that should hold independently of the paper tables.

Purpose: before reproducing paper tables, these cells verify that the simulator behaves correctly in limiting cases where independent reference prices or structural properties are available. These checks are kept in full in the fast demo.


### 5.1 `nu = 0` should reduce SABR to CEV

This isolates the zero vol-of-vol limit. SABR becomes a CEV model, so Monte Carlo call prices can be checked against PyFeng CEV prices.


In [ ]:
params = SABRParams(f0=1.0, sigma0=0.25, nu=0.0, rho=-0.4, beta=0.3)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.6, 1.0, 1.4])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
cev_prices = pf.Cev(sigma=params.sigma0, beta=params.beta, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'cev_price': cev_prices,
    'error': mc_prices - cev_prices,
})


### 5.2 `beta = 1, nu = 0` should reduce SABR to Black-Scholes / lognormal

This checks the lognormal limit. With constant volatility and `beta=1`, the SABR simulator should reproduce Black-Scholes forward call prices.


In [ ]:
params = SABRParams(f0=1.0, sigma0=0.2, nu=0.0, rho=-0.75, beta=1.0)
mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=200_000, seed=12345)
strikes = np.array([0.8, 1.0, 1.2])

terminal = simulate_terminal_forward(params, mc)
mc_prices = np.array([european_call_price(terminal, k) for k in strikes])
bsm_prices = pf.Bsm(sigma=params.sigma0, is_fwd=True).price(strikes, params.f0, mc.maturity)

pd.DataFrame({
    'strike': strikes,
    'mc_price': mc_prices,
    'bsm_price': bsm_prices,
    'error': mc_prices - bsm_prices,
})


### 5.3 Conditional average-variance moment spot check

This is a small diagnostic cell for the `I_t^h` machinery used inside the full simulator.

This inspects the moment inputs used by Algorithm 1. The shifted-lognormal sampler is only as good as the conditional moment calculations it is fitted to.


In [ ]:
sigma_t = np.array([0.2])
sigma_next = np.array([0.24])
nu = 0.4
h = 1.0

mu1, mu2_raw, mu3_raw, mu4_raw = conditional_integrated_variance_moments(sigma_t, sigma_next, nu, h)
mean, var, std, cv, skewness, ex_kurtosis = raw_moments_to_central_stats(mu1, mu2_raw, mu3_raw, mu4_raw)

pd.DataFrame({
    'mu1': mu1,
    'mu2_raw': mu2_raw,
    'mu3_raw': mu3_raw,
    'mu4_raw': mu4_raw,
    'var': var,
    'std': std,
    'cv': cv,
    'skewness': skewness,
    'ex_kurtosis': ex_kurtosis,
})


### 5.4 Martingale sanity check

The forward price should remain close to a martingale: sample means should stay near `F0` up to Monte Carlo noise. This is central to the paper.


In [ ]:
case_v = case_table_3()['Case V']
params = SABRParams(
    f0=case_v['f0'],
    sigma0=case_v['sigma0'],
    nu=case_v['nu'],
    rho=case_v['rho'],
    beta=case_v['beta'],
)

martingale_test(params, maturities=[1, 2, 4, 6, 8, 10], step=1.0, n_paths=30_000, seed0=101)


### 5.5 `|rho| = 1` Islah edge case stability

This is a numerical robustness check for the comparison branch. At full correlation, some generic Islah formulas have removable singularities.


In [ ]:
rows = []
for beta in (0.4, 0.6, 0.8):
    params = SABRParams(f0=1.0, sigma0=0.2, nu=0.2, rho=1.0, beta=beta)
    mc = MonteCarloConfig(maturity=1.0, step=1.0, n_paths=5_000, seed=123)
    terminal = simulate_terminal_forward_islah(params, mc)
    rows.append({
        'beta': beta,
        'all_finite': bool(np.isfinite(terminal).all()),
        'all_nonnegative': bool((terminal >= 0.0).all()),
        'mean_terminal': float(np.mean(terminal)),
        'min_terminal': float(np.min(terminal)),
    })

pd.DataFrame(rows)


## 6. Paper Table 1

By default, this uses the paper table benchmarks embedded in the repo. To compare against the repo PDE benchmark instead, run the later FDM cells or pass benchmark providers in standalone scripts.


Fast demo note: this calls the same Table 1 helper as the full walkthrough, but with smaller `n_paths` and `n_repeats` for live execution.

Purpose: isolate the error from Algorithm 1 in the lognormal SABR case (`beta=1`), where the conditional forward step is analytically simpler.


In [ ]:
table1_df = timed(
    'Paper Table 1 demo-scale run',
    lambda: run_table1_experiment(n_paths=DEMO_N_PATHS_TABLE, n_repeats=DEMO_REPEATS_TABLE, seed0=12345),
)
table1_df


## 7. Paper Table 2


Fast demo note: the full parameter grid is preserved; only the Monte Carlo budget is reduced.

Purpose: separate error behavior across full, partial, and zero correlation. It shows where frozen-coefficient and splitting errors matter more than average-variance sampling error.


In [ ]:
table2_df = timed(
    'Paper Table 2 demo-scale run',
    lambda: run_table2_experiment(n_paths=DEMO_N_PATHS_SMALL, n_repeats=DEMO_REPEATS_SMALL, seed0=12345),
)
table2_df


## 8. Paper Figure 1 Dataset

This generates the skewness and ex-kurtosis comparison data used for the figure. Plotting is optional; the dataset itself is enough for validation.

Purpose: validate the shape of the shifted-lognormal approximation for conditional average variance by comparing skewness and excess kurtosis.


In [ ]:
figure1_df = timed(
    'Paper Figure 1 dataset',
    lambda: figure1_moment_comparison(hat_nu=0.4),
)
figure1_df.head(12)


In [ ]:
ax = figure1_df.plot(x='z_hat', y=['exact_skewness', 'ln_skewness', 'sln_fixed_skewness', 'sln_exact_skewness'], figsize=(10, 4), title='Figure 1 style skewness comparison')
ax.set_ylabel('skewness')


## 9. Paper Table 4

Case I. This includes the simulated rows and the available analytic comparison rows.


Fast demo note: Case I, strike grid, analytic comparison rows, and helper function are unchanged; only simulation budget is reduced.

Purpose: Case I long-maturity strike sweep. This compares the proposed simulation scheme against FDM benchmarks and analytic approximation rows.


In [ ]:
table4_df = timed(
    'Paper Table 4 demo-scale run',
    lambda: run_table4_experiment(n_paths=DEMO_N_PATHS_TABLE, n_repeats=DEMO_REPEATS_TABLE, seed0=12345),
)
table4_df


## 10. Paper Table 5

Case II.


Fast demo note: Case II is unchanged; only simulation budget is reduced.

Purpose: Case II long-maturity strike sweep. It repeats the Table 4 style comparison under a different beta/rho parameter set.


In [ ]:
table5_df = timed(
    'Paper Table 5 demo-scale run',
    lambda: run_table5_experiment(n_paths=DEMO_N_PATHS_TABLE, n_repeats=DEMO_REPEATS_TABLE, seed0=12345),
)
table5_df


## 11. Paper Table 6

Case III. This is the runtime / bias comparison against paper-reference baseline rows.


Fast demo note: Case III and paper-reference baseline rows are unchanged; only our simulation budget is reduced.

Purpose: Case III comparison with other Monte Carlo methods reported in the paper. This section emphasizes speed and bias relative to baseline schemes.


In [ ]:
table6_df = timed(
    'Paper Table 6 demo-scale run',
    lambda: run_table6_experiment(n_paths=DEMO_N_PATHS_TABLE, n_repeats=DEMO_REPEATS_TABLE, seed0=12345),
)
table6_df


## 12. Paper Table 7 / Figure 2

This section can be slow. It reproduces the convergence and runtime trade-off dataset.


Fast demo note: the convergence-study structure is unchanged, but the path ladder and benchmark budget are scaled down for a live run.

Purpose: convergence and runtime tradeoff. The full paper version uses much larger path counts; the fast demo preserves the structure with a smaller path ladder.


In [ ]:
table7_df = timed(
    'Paper Table 7 demo-scale run',
    lambda: run_table7_experiment(
        n_paths_base=DEMO_TABLE7_BASE_PATHS,
        n_repeats=1,
        seed0=12345,
        benchmark_source='mc',
        benchmark_n_paths=DEMO_TABLE7_BENCHMARK_PATHS,
        benchmark_repeats=1,
    ),
)
table7_df


In [ ]:
figure2_df = timed(
    'Paper Figure 2 demo-scale dataset',
    lambda: figure2_runtime_tradeoff(
        n_paths_base=DEMO_TABLE7_BASE_PATHS,
        n_repeats=1,
        seed0=12345,
        benchmark_source='mc',
    ),
)
figure2_df


## 13. Paper Figure 3

This compares the paper scheme against the Islah approximation across maturities.


Fast demo note: the method-vs-Islah comparison is unchanged, but paths/repeats are reduced.

Purpose: direct comparison between the paper method and Islah. The key diagnostic is forward-price drift: the paper method should keep `E[F_T]` closer to `F0`.


In [ ]:
figure3_df = timed(
    'Paper Figure 3 demo-scale run',
    lambda: run_figure3_experiment(
        n_paths=DEMO_FIGURE3_PATHS,
        n_repeats=1,
        seed0=12345,
        benchmark_source='mc',
        benchmark_n_paths=DEMO_FIGURE3_BENCHMARK_PATHS,
        benchmark_repeats=1,
    ),
)
figure3_df.head(20)


In [ ]:
pivot_forward = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='forward_error')
pivot_option = figure3_df.pivot_table(index='maturity', columns=['model', 'step'], values='option_error')

display(pivot_forward)
display(pivot_option)


## 14. Validation Summary

Start with quick mode. The paper-scale run is much slower.


Fast demo note: quick validation is retained. Full paper-scale validation remains commented out because it is intentionally slow.

Purpose: summarize whether the current implementation is structurally consistent with the paper. In fast demo mode this is a smoke test, not a final statistical claim.


In [ ]:
validation_quick = timed(
    'Validation quick smoke test',
    lambda: run_full_validation(quick_mode=True),
)
pd.Series({
    'table1_status': validation_quick['table1_status'],
    'table2_status': validation_quick['table2_status'],
    'overall_conclusion': validation_quick['overall_conclusion'],
    'replication_conclusion': validation_quick['replication_conclusion'],
})


In [ ]:
# Uncomment this only when you want the full validation sweep.
# This is intentionally left paper-scale and can be slow.
# validation_full = run_full_validation(quick_mode=False)
# pd.Series({
#     'table1_status': validation_full['table1_status'],
#     'table2_status': validation_full['table2_status'],
#     'overall_conclusion': validation_full['overall_conclusion'],
#     'replication_conclusion': validation_full['replication_conclusion'],
# })

runtime_df = pd.DataFrame(runtime_rows)
runtime_df.loc[len(runtime_df)] = {'section': 'Notebook elapsed so far', 'seconds': time.perf_counter() - notebook_start}
runtime_df


## 15. Optional CSV Export Helpers

Use these after running the sections you care about.

Purpose: after a live run, uncomment these lines only for the tables or figures you want to export.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'notebook_exports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR


In [ ]:
# Example exports. Uncomment what you need.
# table1_df.to_csv(OUTPUT_DIR / 'table1.csv', index=False)
# table2_df.to_csv(OUTPUT_DIR / 'table2.csv', index=False)
# table4_df.to_csv(OUTPUT_DIR / 'table4.csv', index=False)
# table5_df.to_csv(OUTPUT_DIR / 'table5.csv', index=False)
# table6_df.to_csv(OUTPUT_DIR / 'table6.csv', index=False)
# table7_df.to_csv(OUTPUT_DIR / 'table7.csv', index=False)
# figure1_df.to_csv(OUTPUT_DIR / 'figure1_dataset.csv', index=False)
# figure3_df.to_csv(OUTPUT_DIR / 'figure3_dataset.csv', index=False)
